In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
pip install pygam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.0/522.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 35.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.0
    Uninstalling scipy-1.14.0:
      Successfully uninstalled scipy-1.14.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libpysal 4.9.2 requires packaging>=22, but you have packaging 21.3 which is incompatible.
libpysal 4.9.2 requires shapely>=2.0.1, but you have shapely 1.8.5.post1 which is incompatible.
momepy 0.7.2 requires shapely>=2, but you have shapely 1.8.5.post1 which is incompatible.
pointpats 2.5.0 requires shapely>=2, but you have shapely 1.8.5.post1 which is incompatible.
spaghetti 1.7.6 requires shapely>=2.0.1, but you have 

In [2]:
import pandas as pd
import numpy as np
from pygam import GAM
from pygam.terms import SplineTerm, LinearTerm, FactorTerm, TermList, TensorTerm
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.model_selection import KFold

df = pd.read_csv('/kaggle/input/rechecked-finaldatasetforcausal/finaldatasetwithcourseinformationforcausalanalysis.csv')

df_grouped = df.groupby('id_x').agg({
    'rate': 'mean',
    'group_category': 'first',
    'predicted_aspect': lambda x: list(set(x)),
    'predicted_sentiment_ensemble': lambda x: list(x),
    'price': 'first',
    'is_paid': 'first',
    'num_lectures': 'first'
}).reset_index()


In [3]:

def clean_name(name):
    return name.replace(" ", "_").replace("&", "and")

In [4]:
aspects = df['predicted_aspect'].unique()
for aspect in tqdm(aspects, desc="Creating sentiment columns"):
    clean_aspect = clean_name(aspect)
    sentiment_col = f'sentiment_{clean_aspect}'
    indicator_col = f'has_{clean_aspect}'
    
    df_grouped[sentiment_col] = df_grouped.apply(
        lambda row: np.mean([2 if s == 'Positive' else 1 if s == 'Neutral' else 0 
                             for a, s in zip(row['predicted_aspect'], row['predicted_sentiment_ensemble']) if a == aspect]) 
        if aspect in row['predicted_aspect'] else np.nan, 
        axis=1
    )
    
    df_grouped[indicator_col] = df_grouped[sentiment_col].notna().astype(int)
    df_grouped[sentiment_col] = df_grouped[sentiment_col].fillna(-1)

categories = df_grouped['group_category'].unique()
results = {}

Creating sentiment columns: 100%|██████████| 6/6 [01:34<00:00, 15.68s/it]


In [5]:
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

print("\n=== Results for models with interactions and cross-validation ===")
for category in categories:
    print(f"\nAnalyzing category: {category}")
    df_category = df_grouped[df_grouped['group_category'] == category]
    
    sentiment_columns = [f'sentiment_{clean_name(aspect)}' for aspect in aspects]
    indicator_columns = [f'has_{clean_name(aspect)}' for aspect in aspects]
    continuous_columns = ['price', 'num_lectures']
    binary_columns = ['is_paid']
    
    X = df_category[sentiment_columns + indicator_columns + continuous_columns + binary_columns]
    y = df_category['rate']
    
    scaler = StandardScaler()
    X_continuous_scaled = scaler.fit_transform(X[continuous_columns])
    X_scaled = X.copy()
    X_scaled[continuous_columns] = X_continuous_scaled
    
    r_squared_scores = []
    all_ates = []
    all_interactions = []
    
    for train_index, test_index in kf.split(X_scaled):
        X_train, X_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        main_terms = [
            *[FactorTerm(feature=i) for i in range(len(sentiment_columns))],
            *[LinearTerm(feature=i) for i in range(len(sentiment_columns), len(sentiment_columns) + len(indicator_columns))],
            LinearTerm(feature=len(sentiment_columns) + len(indicator_columns)),  # สำหรับ is_paid
            *[SplineTerm(feature=i, n_splines=10) for i in range(len(sentiment_columns) + len(indicator_columns) + 1, X_scaled.shape[1])]
        ]
        
        interaction_terms = []
        for i in range(len(sentiment_columns)):
            for j in range(i+1, len(sentiment_columns)):
                interaction_terms.append(TensorTerm(FactorTerm(i), FactorTerm(j)))
        
        # รวม main effects และ interaction terms
        all_terms = TermList(*main_terms, *interaction_terms)
        
        gam = GAM(all_terms)
        gam.fit(X_train, y_train)
        r_squared = gam.statistics_['pseudo_r2']['explained_deviance']
        r_squared_scores.append(r_squared)
        
        ates = []
        interactions = []
        for i, col in enumerate(X_test.columns):
            if i < len(sentiment_columns):  
                X_plot = X_test.copy()
                X_plot[col] = -1  # เริ่มจาก -1 แทน 0
                partial_dep = []
                for level in [-1, 0, 1, 2]:  # เพิ่ม -1 เข้าไป
                    X_plot[col] = level
                    pd_level = gam.predict(X_plot).mean()
                    partial_dep.append(pd_level)

                # คำนวณผลต่างระหว่างทุกคู่ที่เกี่ยวข้อง
                diffs = [
                    abs(partial_dep[1] - partial_dep[0]),  # 0 - (-1)
                    abs(partial_dep[2] - partial_dep[1]),  # 1 - 0
                    abs(partial_dep[3] - partial_dep[2]),  # 2 - 1
                    abs((partial_dep[3] - partial_dep[1])/2)  # (2 - 0) / 2
                ]
                ate = np.mean(diffs)
                ates.append((col, ate))
            elif i < len(sentiment_columns) + len(indicator_columns):
                # สำหรับ indicator columns
                X_plot = X_test.copy()
                X_plot[col] = 0
                pd_0 = gam.predict(X_plot).mean()
                X_plot[col] = 1
                pd_1 = gam.predict(X_plot).mean()
                ate = abs(pd_1 - pd_0)
                ates.append((col, ate))
            else: 
                partial_dep = gam.partial_dependence(term=i, X=X_test)
                ate = partial_dep.mean()
                ates.append((col, ate))
        
        all_ates.append(dict(ates))
        
        interaction_ates = []
        for k, term in enumerate(interaction_terms):
            interaction_index = len(main_terms) + k
            partial_dep = gam.partial_dependence(term=interaction_index, X=X_test)
            interaction_effect = np.abs(partial_dep).mean()

            feature1 = sentiment_columns[term.feature[0]]
            feature2 = sentiment_columns[term.feature[1]]
            interaction_ates.append((f"{feature1} x {feature2}", interaction_effect))
        
        all_interactions.append(dict(interaction_ates))
    mean_r_squared = np.mean(r_squared_scores)
    

    mean_ates = {col: np.mean([fold[col] for fold in all_ates]) for col in all_ates[0].keys()}
    

    mean_interactions = {col: np.mean([fold[col] for fold in all_interactions]) for col in all_interactions[0].keys()}
    
    print(f"Mean GAM R-squared: {mean_r_squared:.4f}")
    
    print("\nMean ATEs for Main Effects:")
    ates_df = pd.DataFrame(list(mean_ates.items()), columns=['Feature', 'ATE'])
    ates_df = ates_df.sort_values('ATE', ascending=False)
    print(ates_df.to_string(index=False, float_format='{:.4f}'.format))
    
    print("\nMean ATEs for Interaction Terms:")
    interactions_df = pd.DataFrame(list(mean_interactions.items()), columns=['Interaction', 'ATE'])
    interactions_df = interactions_df.sort_values('ATE', ascending=False)
    print(interactions_df.to_string(index=False, float_format='{:.4f}'.format))
    
    results[category] = {
        'mean_r_squared': mean_r_squared,
        'mean_ates': mean_ates,
        'mean_interactions': mean_interactions
    }
    




=== Results for models with interactions and cross-validation ===

Analyzing category: Technology and IT
Mean GAM R-squared: 0.6568

Mean ATEs for Main Effects:
                          Feature    ATE
             sentiment_Instructor 1.1227
              sentiment_Financial 0.9939
    sentiment_Design_and_Material 0.9454
sentiment_Assignment_and_Tutorial 0.9233
 sentiment_Platform_and_Technical 0.9166
       sentiment_Skill_and_Career 0.8280
                     num_lectures 0.3012
                          is_paid 0.2787
                   has_Instructor 0.1829
             has_Skill_and_Career 0.1592
       has_Platform_and_Technical 0.1237
      has_Assignment_and_Tutorial 0.1198
          has_Design_and_Material 0.1077
                    has_Financial 0.0566
                            price 0.0000

Mean ATEs for Interaction Terms:
                                                         Interaction    ATE
            sentiment_Instructor x sentiment_Assignment_and_Tutorial 0.1